In [1]:
import scenic
import tempfile
import pathlib

/home/luko/.cache/pypoetry/virtualenvs/premise--qB4vHH_-py3.11/lib/python3.11/site-packages/scenic/core/errors.py:271: UserWarning: unable to install sys.excepthook to format Scenic backtraces
  warnings.warn("unable to install sys.excepthook to format Scenic backtraces")


In [2]:
from scenic.simulators.newtonian import NewtonianSimulator

In [7]:
scenario = scenic.scenarioFromFile('../premise/examples/badlyParkedCarPullingIn.scenic',
                                   model='scenic.simulators.newtonian.driving_model',
                                   mode2D=True)

/home/luko/.cache/pypoetry/virtualenvs/premise--qB4vHH_-py3.11/lib/python3.11/site-packages/scenic/formats/opendrive/xodr_parser.py:753: OpenDriveWarning: ignoring shoulder in the middle of road 12
  warn(f"ignoring {name} in the middle of road {self.id_}")
/home/luko/.cache/pypoetry/virtualenvs/premise--qB4vHH_-py3.11/lib/python3.11/site-packages/scenic/formats/opendrive/xodr_parser.py:753: OpenDriveWarning: ignoring shoulder in the middle of road 34
  warn(f"ignoring {name} in the middle of road {self.id_}")
/home/luko/.cache/pypoetry/virtualenvs/premise--qB4vHH_-py3.11/lib/python3.11/site-packages/scenic/formats/opendrive/xodr_parser.py:753: OpenDriveWarning: ignoring shoulder in the middle of road 35
  warn(f"ignoring {name} in the middle of road {self.id_}")
/home/luko/.cache/pypoetry/virtualenvs/premise--qB4vHH_-py3.11/lib/python3.11/site-packages/scenic/formats/opendrive/xodr_parser.py:753: OpenDriveWarning: ignoring shoulder in the middle of road 36
  warn(f"ignoring {name} in 

In [8]:
#parementers - sample set
trace_num = 10
trace_len = 100

#how many rounds of learning
rounds = 5

#parameters - translating scenic data
min = -300
max = 300
step = 150
closeness = 20 
collision_1 = 4.5
collision_2 = 2

#paramenters - intervals, strength 

epsilon = 1/1000

i_nl = 10   #initial lower bound of strength interval 
i_nu = 20   #initial upper bound of strength interval 

i_i_nl = 5  #initial lower bound of strength interval for initial distribution
i_i_nu = 10



In [9]:
all_states = []


for x in range(min, max, step):
    for y in range(min, max, step):
        for z in range(min, max, step):
            for v in range(min, max, step):
                for k in (('close'), ('far')):
                    for j in (('collision'), ('no_collision')):
                        all_states.append([])
                        all_states[-1].append(((x, x+step), (y, y+step)))
                        all_states[-1].append(((z, z+step),(v, v+step)))
                        all_states[-1].append(k)
                        all_states[-1].append(j)
     


In [10]:
samples = []
for n in range(trace_num): 
    scene, _ = scenario.generate()
    simulator = NewtonianSimulator()
    simulation = simulator.simulate(scene, maxSteps=trace_len-1) 
    if simulation:  # `simulate` can return None if simulation fails
        result = simulation.result
        trace = []
        for i, state in enumerate(result.trajectory):
            egoPos, parkedCarPos = state
            state = []

            for x in range(min,max,step):
                 for y in range(min,max,step): 
                     if (x < egoPos[0] <= x+step) and (y < egoPos[1] <=y+step):
                        state.append(((x, x+step),(y, y+step)))
                            
            for z in range(min,max,step):
                for v in range(min,max,step):
                    if (z< parkedCarPos[0] <= z+step) and (v < parkedCarPos[1] <= v+step):
                        state.append(((z, z+step),(v, v+step)))
                             
            if abs((egoPos[0]) - (parkedCarPos[0])) < closeness and abs((egoPos[1]) - (parkedCarPos[1])) < closeness: 
                state.append('close')    
            else: 
                state.append('far')

            if abs(egoPos[0] - parkedCarPos[0]) < 4.5 and abs(egoPos[1] - parkedCarPos[1]) < 2:
                state.append('collision')
                
            else:
                state.append('no_collision')

            
                
            trace.append(state)
        samples.append(trace)


#INITIAL DISTRIBUTION INTERVALS

initial_count = {}

for s in all_states: 
    initial_count[tuple(s)] = 0

for t in samples:
    for s in all_states: 
        if tuple(s) == t[0]: 
            initial_count[tuple(s)] +=1



initial_nl = {}
initial_nu = {}

for s in all_states: 
    initial_nl[tuple(s)] = i_i_nl
    initial_nu[tuple(s)] = i_i_nu
    

initial_interval = {}

for s in all_states:
    initial_interval[tuple(s)] = [epsilon, 1-epsilon]

for n in initial_count.keys():
    for i in initial_interval.keys():
        if n == i:
            if any((initial_count[x]/trace_num) < initial_interval[n][0] for x in  initial_count.keys()): 
                initial_interval[n][0] = (((initial_interval[n][0] * initial_nl[n]) + initial_count[n])/(initial_nl[n] + trace_num))
            else: 
                initial_interval[n][0] = (((initial_interval[n][0] * initial_nu[n]) + initial_count[n])/(initial_nu[n] + trace_num))  

for n in initial_count.keys():
    for i in initial_interval.keys():
        if n == i:
            if any((initial_count[x]/trace_num) > initial_interval[n][1] for x in initial_count.keys()): 
                initial_interval[n][0] = (((initial_interval[n][0] * initial_nl[n]) + initial_count[n])/(initial_nl[n] + trace_num))
            else: 
                initial_interval[n][1] = (((initial_interval[n][1] * initial_nu[n]) + initial_count[n])/(initial_nu[n] + trace_num))

for s in all_states: 
    initial_nl[tuple(s)] += trace_num
    initial_nu[tuple(s)] += trace_num

#TRANSITION PROBABILTY INTERVALS

transition_count = {} 

for t in samples: 
    for s in t[0:trace_len-1]: 
        transition_count[tuple(s)] = 0 

for t in samples: 
    for s in t[0:trace_len-1]: 
         for k in transition_count.keys():
             if s == list(k): 
                 transition_count[tuple(s)] +=1

tau_count = {} 

for a in transition_count.keys():
    for b in transition_count.keys():
        tau_count[a,b] = 0 

for t in samples: 
   for n in range(len(t) -1):
       for a in transition_count.keys():
           for b in transition_count.keys():
               if t[n] == list(a) and t[n+1] == list(b):
                   tau_count[a,b] += 1

interval = {}

for k in tau_count.keys():
    interval[k] = [epsilon, 1-epsilon]

nl = {} 
nu = {} 

for k in tau_count.keys(): 
    nl[k] = i_nl 
    nu[k] = i_nu 
    
for n in transition_count.keys():
    for m in tau_count.keys():
        for i in interval.keys():
            if n == m[0] and m == i:
                if any((tau_count[x]/transition_count[n] < interval[x][0] and x[0] == n) for x in tau_count.keys()):
                    interval[i][0] = ((nl[m] * interval[i][0]) + tau_count[m])/(nl[m] + transition_count[n])
                else: 
                    interval[i][0] = ((nu[m] * interval[i][0]) + tau_count[m])/(nu[m] + transition_count[n])
                    

for n in transition_count.keys():
    for m in tau_count.keys():
        for i in interval.keys():
            if n == m[0] and m == i:
                if any((tau_count[x]/transition_count[n] > interval[x][1] and x[0] == n) for x in tau_count.keys()):
                    interval[i][1] = ((nl[m] * interval[i][1]) + tau_count[m])/(nl[m] + transition_count[n])
                else: 
                    interval[i][1] = ((nu[m] * interval[i][1]) + tau_count[m])/(nu[m] + transition_count[n])



for k in transition_count.keys(): 
    for t in tau_count.keys(): 
        if t[0] == k:
            nl[t] += transition_count[k]
            nu[t] += transition_count[k]



In [11]:
for r in range(rounds-1):
        
    samples = []
    
    for n in range(trace_num): 
        scene, _ = scenario.generate()
        simulator = NewtonianSimulator()
        simulation = simulator.simulate(scene, maxSteps=trace_len-1) 
        if simulation:  # 'simulate' can return None if simulation fails
            result = simulation.result
            trace = []
            for i, state in enumerate(result.trajectory):
                egoPos, parkedCarPos = state
                state = []

                for x in range(min,max,step):
                     for y in range(min,max,step): 
                         if (x < egoPos[0] <=x+step) and (y < egoPos[1] <=y+step):
                            state.append(((x, x+step),(y, y+step)))
                            
                for z in range(min,max,step):
                    for v in range(min,max,step):
                        if (z< parkedCarPos[0] <= z+step) and (v < parkedCarPos[1] <= v+step):
                            state.append(((z, z+step),(v, v+step)))
                             
                if abs(abs(egoPos[0]) - abs(parkedCarPos[0])) < closeness and abs(abs(egoPos[1]) - abs(parkedCarPos[1])) < closeness: 
                    state.append('close') 
                else: 
                    state.append('far')

                if abs(egoPos[0] - parkedCarPos[0]) < 4.5 and abs(egoPos[1] - parkedCarPos[1]) < 2:
                    state.append('collision')
                else:
                    state.append('no_collision')
          
                trace.append(state)
            samples.append(trace)

    
    initial_count = {}

    
    for s in all_states: 
        initial_count[tuple(s)] = 0
        
    for t in samples: 
        for k in initial_count.keys():
            if t[0] == list(k):
                initial_count[k] += 1
                
    
    for n in initial_count.keys():
        for i in initial_interval.keys():
            if n == i:
                if any((initial_count[x]/trace_num) < initial_interval[n][0] for x in  initial_count.keys()): 
                    initial_interval[n][0] = (((initial_interval[n][0] * initial_nl[n]) + initial_count[n])/(initial_nl[n] + trace_num))
                else: 
                    initial_interval[n][0] = (((initial_interval[n][0] * initial_nu[n]) + initial_count[n])/(initial_nu[n] + trace_num)) 


    for n in initial_count.keys():
        for i in initial_interval.keys():
            if n == i:
                if any((initial_count[x]/trace_num) > initial_interval[n][1] for x in initial_count.keys()): 
                    initial_interval[n][0] = (((initial_interval[n][0] * initial_nl[n]) + initial_count[n])/(initial_nl[n] + trace_num))
                else: 
                    initial_interval[n][1] = (((initial_interval[n][1] * initial_nu[n]) + initial_count[n])/(initial_nu[n] + trace_num))



    for s in all_states: 
        initial_nl[tuple(s)] += trace_num
        initial_nu[tuple(s)] += trace_num


    transition_count = {} 

    for t in samples: 
        for s in t[0:trace_len-1]: 
            transition_count[tuple(s)] = 0 

    for t in samples: 
        for s in t[0:trace_len-1]: 
            for k in transition_count.keys():
                if s == list(k): 
                    transition_count[tuple(s)] +=1

    tau_count = {} 


    for a in transition_count.keys():
        for b in transition_count.keys():
            tau_count[a,b] = 0 

    for t in samples: 
       for n in range(len(t) -1):
           for a in transition_count.keys():
               for b in transition_count.keys():
                   if t[n] == list(a) and t[n+1] == list(b):
                       tau_count[a,b] += 1 


    for k in tau_count.keys():
        interval.setdefault(k, [epsilon, 1 - epsilon])

    for k in tau_count.keys():
        nl.setdefault(k, i_nl)
        nu.setdefault(k, i_nu)
        

    #TRANSITION COUNT and TAU COUNT does not include the states from last sample 
    #INTERVAL does 
    
    for n in transition_count.keys():
        for m in tau_count.keys():
            for i in interval.keys():
                if n == m[0] and m == i: #BUT THIS TAKES CARE OF IT 
                    if any((tau_count[x]/transition_count[n] < interval[x][0] and x[0] == n) for x in tau_count.keys()):
                        interval[i][0] = ((nl[m] * interval[i][0]) + tau_count[m])/(nl[m] + transition_count[n])
                    else: 
                        interval[i][0] = ((nu[m] * interval[i][0]) + tau_count[m])/(nu[m] + transition_count[n])

    for n in transition_count.keys():
        for m in tau_count.keys():
            for i in interval.keys():
                if n == m[0] and m == i:
                    if any((tau_count[x]/transition_count[n] > interval[x][1] and x[0] == n) for x in tau_count.keys()):
                        interval[i][1] = ((nl[m] * interval[i][1]) + tau_count[m])/(nl[m] + transition_count[n])
                    else: 
                        interval[i][1] = ((nu[m] * interval[i][1]) + tau_count[m])/(nu[m] + transition_count[n])

    
    for k in tau_count.keys():
       nl.setdefault(k, i_nl)
        
    
    for k in tau_count.keys():
       nu.setdefault(k, i_nu)

    for k in transition_count.keys(): 
        for t in tau_count.keys(): 
            if t[0] == k:
                nl[t] += transition_count[k]
                nu[t] += transition_count[k]
                
    
    

In [12]:
for k in all_states:
    for l in all_states:
        interval.setdefault(((tuple(k)),(tuple(l))), [epsilon, 1 - epsilon])

In [13]:
import numpy as np
np.save('11-12-intervals-20round-alltransitions.npy', interval) 
np.save('11-12-initial_intervals-20rounds-alltransitions.npy', initial_interval)